In [ ]:
import os
import numpy as np
import pandas as pd
from google.colab import drive

# 1. Mount your Google Drive
drive.mount('/content/drive')

# 2. Set the directory path where your CSV files live in Drive
# Update this folder path to match where you uploaded the files in Google Drive:
data_dir = '/content/drive/MyDrive/NSCIMS_Data'  # e.g., '/content/drive/MyDrive/...'

# 3. Load all three datasets
f1 = pd.read_csv(os.path.join(data_dir, 'f1_public_2021.csv'), low_memory=False)
f2 = pd.read_csv(os.path.join(data_dir, 'f2_public_2021.csv'), low_memory=False)
rec = pd.read_csv(os.path.join(data_dir, 'rec_public_2021.csv'), low_memory=False)

# 4. Verify shapes and column counts
print(f"Form I Loaded:        {f1.shape[0]:,} rows x {f1.shape[1]:,} columns")
print(f"Form II Loaded:       {f2.shape[0]:,} rows x {f2.shape[1]:,} columns")
print(f"Record Status Loaded: {rec.shape[0]:,} rows x {rec.shape[1]:,} columns")

In [ ]:
# Record Status (rec_public_2021) Cleaning



# Make sure rec is loaded
rec_clean = rec.copy()


# Clean Calendar Year Columns (Convert float YYYY.0 -> nullable integer Int64)
rec_clean["vital_status_year"] = rec_clean["SVitDate"].astype("Int64")
rec_clean["death_year"] = rec_clean["SDthDt"].astype("Int64")


# Make Standardized Tracking & Vital Status Flags
# SStatCur is the official hierarchical tracking status

rec_clean["SStatCur"] = rec_clean["SStatCur"].astype(str).str.strip()

# Primary mortality indicator (1 = confirmed deceased, 0 = alive/censored)
rec_clean["is_deceased"] = (rec_clean["SStatCur"] == "Deceased").astype(int)

# Active surveillance
rec_clean["is_active_eligible"] = (rec_clean["SStatCur"] == "Eligible").astype(int)

# Attrition flags
# is_lost_to_followup feeds Step 5 in the team's eligibility check
rec_clean["is_lost_to_followup"] = (rec_clean["SStatCur"] == "Lost").astype(int)
rec_clean["is_withdrew_or_untraced"] = rec_clean["SStatCur"].isin(
    ["Withdrew Consent", "ID Unknown"]
).astype(int)

# Clinical recovery (exempt from community follow-up)
rec_clean["is_neuro_recovered"] = rec_clean["SStatCur"].isin(
    ["Norm-Min", "Minimal", "Normal"]
).astype(int)


In [ ]:
# Clean Cause of Death Variables (SCsDth1 - SCsDth5)
# In SCsDth1:
#   '888.8' = Patient Alive (recode to NaN for cause-of-death analyses)
#   '999.9' = Deceased, cause unknown (preserve as "Unknown")

cod_cols = ["SCsDth1", "SCsDth2", "SCsDth3", "SCsDth4", "SCsDth5"]

for col in cod_cols:
    rec_clean[col] = rec_clean[col].astype(str).str.strip()
    rec_clean[col] = rec_clean[col].replace(
        {"888.8": np.nan, "nan": np.nan, "None": np.nan, "": np.nan}
    )


In [ ]:
# Clinical Cause-of-Death Categorization (for Community Integration Impact)
# Maps ICD-10 primary causes into organ-system categories

def categorize_cause_of_death(code):
    if pd.isna(code):
        return np.nan
    c = str(code).upper()
    if c == "999.9":
        return "Unknown / Unspecified"
    elif c.startswith(("A40", "A41")) or c == "B37.7":
        return "Sepsis / Severe Infection"
    elif c.startswith(("J13", "J15", "J18", "J69")):
        return "Pneumonia / Respiratory Infection"
    elif c.startswith(("J96", "J80", "R09.2")):
        return "Acute Respiratory Failure"
    elif c.startswith(("I21", "I25", "I46", "I50")):
        return "Cardiovascular Disease"
    elif c.startswith("I26"):
        return "Pulmonary Embolism"
    elif c.startswith(("I60", "I61", "I64")):
        return "Stroke / Cerebrovascular"
    elif c.startswith(("C", "D")):
        return "Malignant Neoplasms"
    elif c.startswith(("V", "W", "X", "Y")):
        return "External Trauma / Accidents / Suicide"
    else:
        return "Other Medical Conditions"

rec_clean["primary_death_category"] = rec_clean["SCsDth1"].apply(categorize_cause_of_death)

# Confirm cleaned record status matches Form I 1:1
print(f"Record Status cleaned: {len(rec_clean):,} rows, {rec_clean['UniID'].nunique():,} unique patients")
print(f"Confirmed Deceased: {rec_clean['is_deceased'].sum():,} | Active Eligible: {rec_clean['is_active_eligible'].sum():,}")

In [ ]:
def sentinel_codes(series):
    """Return the 7/8/9 sentinel values matching this column's width,
    including decimal sentinels like 88.8 / 99.9 for genuinely
    fractional-valued columns."""
    s = pd.to_numeric(series, errors="coerce").dropna()
    if s.empty:
        return {"declined": pd.NA, "not_applicable": pd.NA, "unknown": pd.NA}

    int_width = max(len(str(abs(int(v)))) for v in s)

    # Correct check: does any value have a non-zero fractional part?
    has_decimal = (s % 1 != 0).any()

    if has_decimal:
        return {
            #"declined": float(f"{'7' * int_width}.7"),
            "not_applicable": float(f"{'8' * int_width}.8"),
            "unknown": float(f"{'9' * int_width}.9"),
        }

    return {
        #"declined": int("7" * int_width),
        "not_applicable": int("8" * int_width),
        "unknown": int("9" * int_width),
    }
  # For some variables 7 represents valid code, decided to exclude it from the sentinel code cleaning

In [ ]:
## Check APResInj ( to see if it removed the 7 value)
## AFScorRb & AFScorDs, 88 is a valid value
## AEducLvl, 8 is valid value
## ANCatAdm ANCatRhb ANCatDis, 8 is a valid value

col_to_leave = ["AFScorRb", "AFScorDs", "AEducLvl", "ANCatAdm", "ANCatRhb", "ANCatDis"]

In [ ]:
def clean_column(series):
    codes = sentinel_codes(series)
    return series.replace(list(codes.values()), pd.NA)

form1_cols = f1.columns.tolist()

f1_clean = f1.copy()

for col in form1_cols:
    if col in col_to_leave:
        continue
    if pd.api.types.is_numeric_dtype(f1[col]):
        codes = sentinel_codes(f1[col])
        f1_clean[col] = f1[col].replace(list(codes.values()), pd.NA)

In [ ]:
FIXED_DECIMAL_SENTINELS = [777.77, 888.88, 999.99, 999.9]

# Only includes columns I've actually confirmed carry this pattern
confirmed_decimal_sentinel_cols = [
    "ARsTScr", "ARsStEr"
    # add others here once verified via the check above
]

for col in confirmed_decimal_sentinel_cols:
    f1_clean[col] = f1_clean[col].replace(FIXED_DECIMAL_SENTINELS, np.nan)

In [ ]:
# AExtCsIj is categorical (ICD-10 external-cause-of-injury codes), not continuous.
# 999.999 is the "unknown cause" sentinel - recode explicitly rather than to NaN,
# since "Unknown cause" and "not collected" are different things worth keeping separate.

f1_clean["AExtCsIj"] = f1_clean["AExtCsIj"].astype(str).str.strip()
f1_clean["AExtCsIj"] = f1_clean["AExtCsIj"].replace(
    {"999.999": "Unknown"}
)

# Confirm the recode landed and see what's left
f1_clean["AExtCsIj"].value_counts(dropna=False).head(20)

In [ ]:
# Clean Form II (f2_clean)


f2_clean = f2.copy()

# 1. Columns to protect from generic sentinel stripping
# (Variables where 8, 88, 7, etc. are meaningful category codes or where
f2_col_to_leave = [
    "UniID",
    "BYear",
    "BFolUpCt",     # 5 = Lost to follow-up, 8 = Extended follow-up
    "BLostRsn",     # Specific reason codes for attrition
    "BMdTrnsp",     # 1-8 are valid transport modes
    "BDrvModV",     # 0=No, 1=Yes, 8=N/A
    "BRhspNbr",     # 8=Unknown count (rehospitalized)
]

# 2. Apply sentinel codes replacement across numeric Form II columns
for col in f2_clean.columns:
    if col in f2_col_to_leave:
        continue
    if pd.api.types.is_numeric_dtype(f2_clean[col]):
        codes = sentinel_codes(f2_clean[col])
        f2_clean[col] = f2_clean[col].replace(list(codes.values()), pd.NA)

# 3. Clean decimal sentinels identified in team notes
f2_confirmed_decimal_cols = ["BRsTScr", "BRsStEr"]
for col in f2_confirmed_decimal_cols:
    if col in f2_clean.columns:
        f2_clean[col] = f2_clean[col].replace(FIXED_DECIMAL_SENTINELS, np.nan)

# 4. Clean undocumented sentinel code in PHQ-9 (BBPHQSDS) noted by Pair 3
if "BBPHQSDS" in f2_clean.columns:
    f2_clean["BBPHQSDS"] = f2_clean["BBPHQSDS"].replace([77, 88, 99], np.nan)

print(f"✓ f2_clean defined: {len(f2_clean):,} rows x {f2_clean.shape[1]} columns")

## Anuoluwa Data joining and inclusion criteria

In [ ]:
# Form I + Record Status: 1:1 merge on UniID
baseline = f1_clean.merge(rec_clean, on="UniID", how="left", validate="1:1")

# Attach baseline info to every Form II row for that person (many-to-one)
full = f2_clean.merge(baseline, on="UniID", how="left", validate="m:1")

In [ ]:
# ---------------------------------------------------------------
# Step 4: Inclusion criteria — exclusion code definitions
# ---------------------------------------------------------------
APRESDIS_EXCLUDE = [7, 8, 99]          # Deceased, Other/unclassified, Unknown
APRESDIS_ASSISTED_LIVING_CODE = 10      # included by default — toggle for sensitivity check
BPLCRES_EXCLUDE = [8, 99]               # Other/unclassified, Unknown
BRHSPNBR_EXCLUDE = [9]                  # Unknown
BRHSPNBR_COUNT_UNKNOWN_CODE = 8         # rehospitalized, count unknown — kept as valid flag

# ---------------------------------------------------------------
# Step 5: Row-level eligibility flags
# ---------------------------------------------------------------
full["valid_discharge_loc"] = ~full["APResDis"].isin(APRESDIS_EXCLUDE)
full["is_assisted_living_discharge"] = full["APResDis"] == APRESDIS_ASSISTED_LIVING_CODE

blank_bad_living = full["BPlcRes"].isna() & (~full["is_lost_to_followup"])
full["valid_living_loc"] = (~full["BPlcRes"].isin(BPLCRES_EXCLUDE)) & (~blank_bad_living)
full["unexpected_blank_living_loc"] = blank_bad_living

blank_bad_rehosp = full["BRhspNbr"].isna() & (~full["is_lost_to_followup"])
full["valid_rehosp_status"] = (~full["BRhspNbr"].isin(BRHSPNBR_EXCLUDE)) & (~blank_bad_rehosp)
full["rehosp_count_unknown"] = full["BRhspNbr"] == BRHSPNBR_COUNT_UNKNOWN_CODE
full["unexpected_blank_rehosp"] = blank_bad_rehosp

# ---------------------------------------------------------------
# Step 6: Row-level construct validity — pooled across variables
# ---------------------------------------------------------------
sdoh_vars = ["BPlcRes", "BMarStat", "BEducLvl", "BPrLvlSt", "BFmIncLv", "BPrimPay", "BMdTrnsp"]

# NOTE: PHQ (BBPHQMDS, BBPHQSDS) folded into functional ability per user decision,
# alongside self-care (BSFSCRS) and physical independence (BCHPITot).
# "Pooled" means: a year counts as a valid timepoint if ANY one of the listed
# variables is non-missing that year — it does not require the same specific
# variable to repeat across years.
functional_vars = ["BSFSCRS", "BCHPITot", "BBPHQMDS", "BBPHQSDS"]

full["sdoh_row_valid"] = full[sdoh_vars].notna().any(axis=1) & full["valid_living_loc"]
full["functional_row_valid"] = full[functional_vars].notna().any(axis=1)

# ---------------------------------------------------------------
# Step 7: Collapse to person-level eligibility
# ---------------------------------------------------------------
person_summary = full.groupby("UniID").agg(
    n_sdoh_timepoints=("sdoh_row_valid", "sum"),
    n_functional_timepoints=("functional_row_valid", "sum"),
    baseline_discharge_valid=("valid_discharge_loc", "first"),
    has_valid_living_loc=("valid_living_loc", "any"),
    has_valid_rehosp=("valid_rehosp_status", "any"),
).reset_index()

person_summary["meets_sdoh_criterion"] = person_summary["n_sdoh_timepoints"] >= 2
person_summary["meets_functional_criterion"] = person_summary["n_functional_timepoints"] >= 2
person_summary["eligible"] = (
    person_summary["baseline_discharge_valid"]
    & person_summary["has_valid_living_loc"]
    & person_summary["has_valid_rehosp"]
    & person_summary["meets_sdoh_criterion"]
    & person_summary["meets_functional_criterion"]
)

# ---------------------------------------------------------------
# Step 8: Transparency funnel
# ---------------------------------------------------------------
funnel = pd.Series({
    "Total unique people (Form I)": f1_clean["UniID"].nunique(),
    "Valid discharge location": person_summary["baseline_discharge_valid"].sum(),
    "+ valid living location & rehosp status": (
        person_summary["baseline_discharge_valid"]
        & person_summary["has_valid_living_loc"]
        & person_summary["has_valid_rehosp"]
    ).sum(),
    "+ 2 or more SDOH timepoints": (
        person_summary["baseline_discharge_valid"]
        & person_summary["has_valid_living_loc"]
        & person_summary["has_valid_rehosp"]
        & person_summary["meets_sdoh_criterion"]
    ).sum(),
    "+ 2 or more functional timepoints (final eligible)": person_summary["eligible"].sum(),
})
print(funnel)
print("\nUnexpected blank living location rows:", full["unexpected_blank_living_loc"].sum())
print("Unexpected blank rehosp rows:", full["unexpected_blank_rehosp"].sum())

# ---------------------------------------------------------------
# Step 9: Final analysis-ready cohort
# ---------------------------------------------------------------
eligible_ids = person_summary.loc[person_summary["eligible"], "UniID"]
analysis_df = full[full["UniID"].isin(eligible_ids)].copy()

print(f"\nFinal analysis cohort: {analysis_df['UniID'].nunique()} people, {len(analysis_df)} person-year rows")

#Cross-Dataset Linkage Architecture

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

# 1:1 Linkage: Baseline Clinical Data (Form I) + Vital Tracking (Record Status)
baseline_linked = pd.merge(f1_clean, rec_clean, on="UniID", how="inner")

# Many:1 Linkage: Longitudinal Outcomes (Form II) + Baseline & Vital Tracking
full_linked = pd.merge(f2_clean, baseline_linked, on="UniID", how="left")

print(f"Baseline Cohort Linked (1:1): {len(baseline_linked):,} rows (35,675 unique patients)")
print(f"Longitudinal Records Linked (M:1): {len(full_linked):,} person-year observations")

##Who Returned Home After Rehabilitation

In [ ]:
# Define analytic cohort (exclude in-hospital deaths APResDis == 7 and unknown == 99)
home_df = baseline_linked[~baseline_linked["APResDis"].isin([7, 99])].copy()
home_df["returned_home"] = (home_df["APResDis"] == 1).astype(int)

# Group Summary
home_summary = home_df.groupby("returned_home").agg(
    n=("UniID", "count"),
    mean_age=("AInjAge", "mean"),
    median_age=("AInjAge", "median"),
    median_rehab_los=("AHDaSyRb", "median"),
    median_fim_motor=("AFScorDs", "median"),
    pct_male=("ASex", lambda x: (x == 1).mean() * 100)
).reset_index()

# Statistical Tests
u_age, p_age = stats.mannwhitneyu(
    home_df[home_df["returned_home"] == 1]["AInjAge"].dropna(),
    home_df[home_df["returned_home"] == 0]["AInjAge"].dropna()
)
u_fim, p_fim = stats.mannwhitneyu(
    home_df[home_df["returned_home"] == 1]["AFScorDs"].dropna(),
    home_df[home_df["returned_home"] == 0]["AFScorDs"].dropna()
)
chi2_injury, p_injury, _, _ = stats.chi2_contingency(
    pd.crosstab(home_df["ANCatDis"], home_df["returned_home"])
)

# Cross-tabulation by Injury Subgroup
subgroup_home = pd.crosstab(
    home_df["ANCatDis"], home_df["returned_home"], normalize="index"
) * 100

##Who Was Rehospitalized

In [ ]:
# Restrict to completed Form II interviews with valid rehospitalization data
f2_valid = full_linked[full_linked["BFolUpCt"].isin([1, 2, 4]) & full_linked["BRhspNbr"].notna()].copy()
f2_valid["is_rehospitalized"] = (f2_valid["BRhspNbr"] > 0).astype(int)

# 1. Rehospitalization Rate by Post-Injury Anniversary Year (BYear)
annual_rehosp = f2_valid.groupby("BYear")["is_rehospitalized"].agg(
    total_interviews="count",
    rehosp_count="sum",
    pct_rehosp=lambda x: x.mean() * 100
).reset_index()

# 2. Most Common Diagnostic Reasons (BRhspRs1)
reasons = f2_valid[f2_valid["is_rehospitalized"] == 1]["BRhspRs1"].value_counts(normalize=True) * 100

# 3. Association with Subsequent Mortality in Record Status
mort_by_rehosp = f2_valid.groupby("is_rehospitalized")["is_deceased"].mean() * 100

##Who Was Unable to Drive

In [ ]:
# Analyze Year 1 adult cohort (Age >= 18)
drv_df = f2_valid[(f2_valid["BYear"] == 1) & (f2_valid["AInjAge"] >= 18)].copy()

# Define driving ability per Data Dictionary
drv_df["able_to_drive"] = drv_df["BMdTrnsp"].isin([1, 2]).astype(int)

# Cross-tabulation by Neurologic Injury Group
drv_by_injury = pd.crosstab(
    drv_df["ANCatDis"], drv_df["able_to_drive"], normalize="index"
) * 100

# Association with Payer / Socioeconomic Access
drv_by_payer = pd.crosstab(
    drv_df["BPrimPay"], drv_df["able_to_drive"], normalize="index"
) * 100
chi2_drv, p_drv, _, _ = stats.chi2_contingency(pd.crosstab(drv_df["ANCatDis"], drv_df["able_to_drive"]))